In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

np.random.seed(42)

n = 5000

data = pd.DataFrame({
    "tenure": np.random.randint(1, 73, n),
    "monthly_charges": np.random.uniform(20, 120, n),
    "total_charges": np.random.uniform(100, 8000, n),
    "contract": np.random.choice(
        ["Month-to-month", "One year", "Two year"], n
    ),
    "internet_service": np.random.choice(
        ["DSL", "Fiber optic", "No"], n
    ),
    "payment_method": np.random.choice(
        ["Electronic check", "Bank transfer", "Credit card"], n
    ),
    "support_calls": np.random.randint(0, 10, n),
    "tech_support": np.random.choice(["Yes", "No"], n)
})

data.head(10)

,tenure,monthly_charges,total_charges,contract,internet_service,payment_method,support_calls,tech_support
0,52,118.950533,1420.361087,One year,DSL,Bank transfer,3,Yes
1,15,88.431425,3316.672535,Two year,No,Bank transfer,0,No
2,72,114.898067,1934.959582,Month-to-month,DSL,Credit card,5,No
3,61,34.255656,4324.927718,Two year,No,Credit card,7,Yes
4,21,58.213947,1670.745389,One year,Fiber optic,Credit card,3,No
5,24,75.473180,4116.407215,Month-to-month,DSL,Electronic check,0,No
6,3,27.675806,1343.290267,Two year,DSL,Credit card,4,No
7,22,20.418744,5400.700205,One year,No,Bank transfer,9,Yes
8,53,87.043433,6056.107707,Two year,Fiber optic,Credit card,4,No
9,2,84.181952,972.114451,Two year,DSL,Electronic check,1,No


In [3]:
risk_score = (
    (data["tenure"] < 12).astype(int) * 2
    + (data["monthly_charges"] > 80).astype(int) * 2
    + (data["contract"] == "Month-to-month").astype(int) * 3
    + (data["support_calls"] > 5).astype(int) * 2
    + (data["tech_support"] == "No").astype(int)
)

probability = 1 / (1 + np.exp(-(risk_score - 4)))

data["churn"] = (
    np.random.random(n) < probability
).astype(int)

data.head()

,tenure,monthly_charges,total_charges,contract,internet_service,payment_method,support_calls,tech_support,churn
0,52,118.950533,1420.361087,One year,DSL,Bank transfer,3,Yes,0
1,15,88.431425,3316.672535,Two year,No,Bank transfer,0,No,0
2,72,114.898067,1934.959582,Month-to-month,DSL,Credit card,5,No,1
3,61,34.255656,4324.927718,Two year,No,Credit card,7,Yes,0
4,21,58.213947,1670.745389,One year,Fiber optic,Credit card,3,No,0


In [4]:
X = data.drop("churn", axis=1)
y = data["churn"]

categorical_columns = [
    "contract",
    "internet_service",
    "payment_method",
    "tech_support"
]

numeric_columns = [
    "tenure",
    "monthly_charges",
    "total_charges",
    "support_calls"
]

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numeric",
            "passthrough",
            numeric_columns
        )
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)

predictions = pipeline.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.784

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       592
           1       0.76      0.68      0.72       408

    accuracy                           0.78      1000
   macro avg       0.78      0.77      0.77      1000
weighted avg       0.78      0.78      0.78      1000



In [7]:
joblib.dump(
    pipeline,
    "churn_model.pkl"
)

print("Model saved successfully!")

Model saved successfully!
